# Round 5 — Galaxy Sounds Recorders

Products (position limit 10 each):
- `GALAXY_SOUNDS_DARK_MATTER`
- `GALAXY_SOUNDS_BLACK_HOLES`
- `GALAXY_SOUNDS_PLANETARY_RINGS`
- `GALAXY_SOUNDS_SOLAR_WINDS`
- `GALAXY_SOUNDS_SOLAR_FLAMES`

In [28]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), ""))
from plotter import Plotter, LogVisualizer
import pandas as pd
import plotly.graph_objects as go

DATA_DIR = "../../data/round5"
days = [2, 3, 4]
PRODUCTS = [
    "GALAXY_SOUNDS_DARK_MATTER",
    "GALAXY_SOUNDS_BLACK_HOLES",
    "GALAXY_SOUNDS_PLANETARY_RINGS",
    "GALAXY_SOUNDS_SOLAR_WINDS",
    "GALAXY_SOUNDS_SOLAR_FLAMES",
]
POSITION_LIMIT = 10

In [29]:
raw_prices = pd.concat(
    [pd.read_csv(f"{DATA_DIR}/prices_round_5_day_{d}.csv", delimiter=";") for d in days],
    ignore_index=True,
)
all_prices = raw_prices[raw_prices["product"].isin(PRODUCTS)].reset_index(drop=True)

frames = {p: all_prices[all_prices["product"] == p].reset_index(drop=True) for p in PRODUCTS}
for p, df in frames.items():
    print(f"{p}: {len(df)} rows")

GALAXY_SOUNDS_DARK_MATTER: 30000 rows
GALAXY_SOUNDS_BLACK_HOLES: 30000 rows
GALAXY_SOUNDS_PLANETARY_RINGS: 30000 rows
GALAXY_SOUNDS_SOLAR_WINDS: 30000 rows
GALAXY_SOUNDS_SOLAR_FLAMES: 30000 rows


In [30]:
raw_trades = pd.concat(
    [pd.read_csv(f"{DATA_DIR}/trades_round_5_day_{d}.csv", delimiter=";")
       .assign(day=d)
     for d in days],
    ignore_index=True,
).rename(columns={"symbol": "product"})

all_trades = raw_trades[raw_trades["product"].isin(PRODUCTS)].reset_index(drop=True)
all_trades = all_trades.merge(all_prices, on=["day", "timestamp", "product"], how="left")

trades_by_product = {p: all_trades[all_trades["product"] == p].reset_index(drop=True) for p in PRODUCTS}
for p, df in trades_by_product.items():
    print(f"{p}: {len(df)} trades")

GALAXY_SOUNDS_DARK_MATTER: 733 trades
GALAXY_SOUNDS_BLACK_HOLES: 733 trades
GALAXY_SOUNDS_PLANETARY_RINGS: 733 trades
GALAXY_SOUNDS_SOLAR_WINDS: 733 trades
GALAXY_SOUNDS_SOLAR_FLAMES: 733 trades


## Spread distribution

In [31]:
for p, df in frames.items():
    spread = df["ask_price_1"] - df["bid_price_1"]
    print(f"\n{p} spread:")
    print(spread.describe().to_string())


GALAXY_SOUNDS_DARK_MATTER spread:
count    30000.000000
mean        13.050833
std          1.320217
min          5.000000
25%         13.000000
50%         13.000000
75%         14.000000
max         15.000000

GALAXY_SOUNDS_BLACK_HOLES spread:
count    30000.000000
mean        14.512767
std          1.789342
min          5.000000
25%         14.000000
50%         14.000000
75%         16.000000
max         18.000000

GALAXY_SOUNDS_PLANETARY_RINGS spread:
count    30000.000000
mean        13.690033
std          1.601741
min          5.000000
25%         13.000000
50%         14.000000
75%         15.000000
max         16.000000

GALAXY_SOUNDS_SOLAR_WINDS spread:
count    30000.000000
mean        13.301267
std          1.432471
min          5.000000
25%         13.000000
50%         14.000000
75%         14.000000
max         15.000000

GALAXY_SOUNDS_SOLAR_FLAMES spread:
count    30000.000000
mean        14.071533
std          1.448454
min          6.000000
25%         14.000000
50%   

## Mid price overview

In [53]:
DAY_OFFSET = 1_000_000
fig = go.Figure()
for p, df in frames.items():
    df = df.sort_values(["day", "timestamp"]).copy()
    df["global_ts"] = df["timestamp"] + (df["day"] - df["day"].min()) * DAY_OFFSET
    fig.add_trace(go.Scattergl(x=df["global_ts"], y=df["mid_price"], mode="lines", name=p))
fig.update_layout(title="Galaxy Sounds — mid prices", xaxis_title="global timestamp", yaxis_title="mid price")
fig.show(renderer="browser")

## Per-product orderbook

In [55]:
plt = Plotter(
    [f"{DATA_DIR}/prices_round_5_day_{d}.csv" for d in days],
    [f"{DATA_DIR}/trades_round_5_day_{d}.csv" for d in days],
)
# restrict the plotter to the galaxy sounds products only
plt.prices = plt.prices[plt.prices["product"].isin(PRODUCTS)].reset_index(drop=True)
plt.trades = plt.trades[plt.trades["product"].isin(PRODUCTS)].reset_index(drop=True)
plt.products = [p for p in PRODUCTS if p in set(plt.prices["product"].unique())]

plt.visualize_orderbook(renderer="browser")

interactive(children=(Dropdown(description='Product:', options=('GALAXY_SOUNDS_DARK_MATTER', 'GALAXY_SOUNDS_BL…

## Cointegration tests (Engle-Granger)

In [17]:
# from statsmodels.tsa.stattools import coint
# from itertools import combinations
# import numpy as np

# # align mid prices on (day, timestamp) so all series have the same index
# mid_wide = (
#     all_prices[["day", "timestamp", "product", "mid_price"]]
#     .pivot_table(index=["day", "timestamp"], columns="product", values="mid_price")
#     .sort_index()
#     .dropna()
# )
# print(f"aligned rows: {len(mid_wide)}")

# results = []
# for a, b in combinations(PRODUCTS, 2):
#     if a not in mid_wide.columns or b not in mid_wide.columns:
#         continue
#     t_stat, p_value, crit = coint(mid_wide[a], mid_wide[b])
#     results.append({
#         "pair": f"{a} | {b}",
#         "t_stat": t_stat,
#         "p_value": p_value,
#         "crit_1%": crit[0],
#         "crit_5%": crit[1],
#         "crit_10%": crit[2],
#         "cointegrated_5%": p_value < 0.05,
#     })

# coint_df = pd.DataFrame(results).sort_values("p_value").reset_index(drop=True)
# coint_df

aligned rows: 30000


,pair,t_stat,p_value,crit_1%,crit_5%,crit_10%,cointegrated_5%
0,GALAXY_SOUNDS_DARK_MATTER | GALAXY_SOUNDS_PLAN...,-2.980366,0.114947,-3.896805,-3.336334,-3.044591,False
1,GALAXY_SOUNDS_DARK_MATTER | GALAXY_SOUNDS_SOLA...,-2.729142,0.189273,-3.896805,-3.336334,-3.044591,False
2,GALAXY_SOUNDS_DARK_MATTER | GALAXY_SOUNDS_SOLA...,-2.722776,0.191494,-3.896805,-3.336334,-3.044591,False
3,GALAXY_SOUNDS_DARK_MATTER | GALAXY_SOUNDS_BLAC...,-2.722106,0.191729,-3.896805,-3.336334,-3.044591,False
4,GALAXY_SOUNDS_SOLAR_WINDS | GALAXY_SOUNDS_SOLA...,-2.269827,0.388701,-3.896805,-3.336334,-3.044591,False
5,GALAXY_SOUNDS_PLANETARY_RINGS | GALAXY_SOUNDS_...,-1.045136,0.893947,-3.896805,-3.336334,-3.044591,False
6,GALAXY_SOUNDS_PLANETARY_RINGS | GALAXY_SOUNDS_...,-1.039004,0.895196,-3.896805,-3.336334,-3.044591,False
7,GALAXY_SOUNDS_BLACK_HOLES | GALAXY_SOUNDS_SOLA...,-0.873117,0.924393,-3.896805,-3.336334,-3.044591,False
8,GALAXY_SOUNDS_BLACK_HOLES | GALAXY_SOUNDS_SOLA...,-0.670716,0.949707,-3.896805,-3.336334,-3.044591,False
9,GALAXY_SOUNDS_BLACK_HOLES | GALAXY_SOUNDS_PLAN...,-0.370513,0.972373,-3.896805,-3.336334,-3.044591,False


## All products combined view

All 5 products on one chart — mid + bids + asks + trades. Click legend entries to toggle individual traces, or double-click to isolate. Traces are grouped by product so toggling a group hides all of its lines/markers.

In [34]:
import plotly.colors as pc

DAY_OFFSET = 1_000_000
colors = pc.qualitative.Plotly  # one color per product

fig = go.Figure()

for i, product in enumerate(PRODUCTS):
    color = colors[i % len(colors)]
    df = frames[product].sort_values(["day", "timestamp"]).copy()
    df["global_ts"] = df["timestamp"] + (df["day"] - df["day"].min()) * DAY_OFFSET

    # mid price (visible by default)
    fig.add_trace(go.Scattergl(
        x=df["global_ts"], y=df["mid_price"],
        mode="lines", name=f"{product} mid",
        legendgroup=product, line=dict(color=color, width=2),
    ))

    # bid/ask levels (hidden by default — click legend to show)
    for lvl in range(1, 4):
        fig.add_trace(go.Scattergl(
            x=df["global_ts"], y=df[f"bid_price_{lvl}"],
            mode="lines", name=f"{product} bid_{lvl}",
            legendgroup=product, visible="legendonly",
            line=dict(color=color, width=1, dash="dash"),
        ))
        fig.add_trace(go.Scattergl(
            x=df["global_ts"], y=df[f"ask_price_{lvl}"],
            mode="lines", name=f"{product} ask_{lvl}",
            legendgroup=product, visible="legendonly",
            line=dict(color=color, width=1, dash="dot"),
        ))

    # trades (visible by default, smaller markers)
    tr = trades_by_product[product].sort_values(["day", "timestamp"]).copy()
    if not tr.empty:
        tr["global_ts"] = tr["timestamp"] + (tr["day"] - tr["day"].min()) * DAY_OFFSET
        sizes = 4 + 8 * (tr["quantity"] / max(tr["quantity"].max(), 1))
        fig.add_trace(go.Scattergl(
            x=tr["global_ts"], y=tr["price"],
            mode="markers", name=f"{product} trades",
            legendgroup=product,
            marker=dict(color=color, size=sizes, symbol="circle", opacity=0.6,
                        line=dict(width=0.5, color="white")),
            customdata=tr["quantity"],
            hovertemplate=f"{product}<br>price: %{{y}}<br>qty: %{{customdata}}<br>ts: %{{x}}<extra></extra>",
        ))

fig.update_layout(
    title="Galaxy Sounds — all products (mid + trades; bids/asks hidden, click legend to show)",
    xaxis_title="global timestamp",
    yaxis_title="price",
    hovermode="closest",
    legend=dict(groupclick="togglegroup"),
    height=700,
)
fig.show(renderer="browser")

In [36]:
trades_by_product['GALAXY_SOUNDS_SOLAR_WINDS'].head()

,timestamp,buyer,seller,product,currency,price,quantity,day,bid_price_1,bid_volume_1,...,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss
0,1700,NaN,NaN,GALAXY_SOUNDS_SOLAR_WINDS,XIRECS,10084.0,4,2,10084,24,...,NaN,NaN,10097,24,10099.0,30.0,NaN,NaN,10090.5,0.0
1,14500,NaN,NaN,GALAXY_SOUNDS_SOLAR_WINDS,XIRECS,9926.0,1,2,9913,13,...,NaN,NaN,9926,13,9927.0,33.0,NaN,NaN,9919.5,0.0
2,15100,NaN,NaN,GALAXY_SOUNDS_SOLAR_WINDS,XIRECS,9936.0,2,2,9923,24,...,NaN,NaN,9936,24,9937.0,36.0,NaN,NaN,9929.5,0.0
3,26500,NaN,NaN,GALAXY_SOUNDS_SOLAR_WINDS,XIRECS,9924.0,4,2,9911,17,...,NaN,NaN,9924,17,9925.0,31.0,NaN,NaN,9917.5,0.0
4,36400,NaN,NaN,GALAXY_SOUNDS_SOLAR_WINDS,XIRECS,9816.0,4,2,9816,21,...,NaN,NaN,9829,21,9830.0,27.0,NaN,NaN,9822.5,0.0


In [54]:
trades_by_product['GALAXY_SOUNDS_BLACK_HOLES']['quantity'].mean()

np.float64(2.4624829467939975)

In [38]:
# check if (timestamp, quantity) of trades are identical across all 5 products
ref = PRODUCTS[0]
ref_df = trades_by_product[ref][["timestamp", "quantity"]].sort_values(["timestamp", "quantity"]).reset_index(drop=True)

print(f"reference: {ref} — {len(ref_df)} trades")
for p in PRODUCTS[1:]:
    df = trades_by_product[p][["timestamp", "quantity"]].sort_values(["timestamp", "quantity"]).reset_index(drop=True)
    same_len = len(df) == len(ref_df)
    same_ts = same_len and (df["timestamp"].values == ref_df["timestamp"].values).all()
    same_qty = same_len and (df["quantity"].values == ref_df["quantity"].values).all()
    print(f"  {p}: rows={len(df)} same_len={same_len} same_ts={same_ts} same_qty={same_qty}")


reference: GALAXY_SOUNDS_DARK_MATTER — 733 trades
  GALAXY_SOUNDS_BLACK_HOLES: rows=733 same_len=True same_ts=True same_qty=True
  GALAXY_SOUNDS_PLANETARY_RINGS: rows=733 same_len=True same_ts=True same_qty=True
  GALAXY_SOUNDS_SOLAR_WINDS: rows=733 same_len=True same_ts=True same_qty=True
  GALAXY_SOUNDS_SOLAR_FLAMES: rows=733 same_len=True same_ts=True same_qty=True


In [39]:
# classify each trade by side using merged bid/ask, then compare across products
def classify(df):
    side = pd.Series("mid", index=df.index)
    side[df["price"] >= df["ask_price_1"]] = "buy"
    side[df["price"] <= df["bid_price_1"]] = "sell"
    return df.assign(side=side)[["timestamp", "side"]].sort_values("timestamp").reset_index(drop=True)

ref = PRODUCTS[0]
ref_df = classify(trades_by_product[ref])
print(f"reference: {ref} — sides: {ref_df['side'].value_counts().to_dict()}")

for p in PRODUCTS[1:]:
    df = classify(trades_by_product[p])
    same_len = len(df) == len(ref_df)
    same_ts = same_len and (df["timestamp"].values == ref_df["timestamp"].values).all()
    same_side = same_len and (df["side"].values == ref_df["side"].values).all()
    mismatches = 0 if same_side else int((df["side"].values != ref_df["side"].values).sum()) if same_len else "n/a"
    print(f"  {p}: same_ts={same_ts} same_side={same_side} mismatches={mismatches}")


reference: GALAXY_SOUNDS_DARK_MATTER — sides: {'sell': 375, 'buy': 358}
  GALAXY_SOUNDS_BLACK_HOLES: same_ts=True same_side=True mismatches=0
  GALAXY_SOUNDS_PLANETARY_RINGS: same_ts=True same_side=True mismatches=0
  GALAXY_SOUNDS_SOLAR_WINDS: same_ts=True same_side=True mismatches=0
  GALAXY_SOUNDS_SOLAR_FLAMES: same_ts=True same_side=True mismatches=0


In [40]:
# imagine one trader did every trade across all 5 products — compute their pnl
def classify(df):
    side = pd.Series("mid", index=df.index)
    side[df["price"] >= df["ask_price_1"]] = "buy"
    side[df["price"] <= df["bid_price_1"]] = "sell"
    return df.assign(side=side)

per_product_pnl = {}
total_pnl = 0.0
for p in PRODUCTS:
    df = classify(trades_by_product[p])
    # cash flow per trade: -price*qty if buy (cash out), +price*qty if sell (cash in)
    cash = (df.loc[df["side"] == "sell", "price"] * df.loc[df["side"] == "sell", "quantity"]).sum() \
         - (df.loc[df["side"] == "buy",  "price"] * df.loc[df["side"] == "buy",  "quantity"]).sum()
    # closing inventory marked at last mid
    net_qty = df.loc[df["side"] == "buy", "quantity"].sum() - df.loc[df["side"] == "sell", "quantity"].sum()
    last_mid = frames[p]["mid_price"].dropna().iloc[-1]
    pnl = cash + net_qty * last_mid
    per_product_pnl[p] = pnl
    total_pnl += pnl
    print(f"{p}: cash={cash:+,.2f}  net_qty={net_qty:+d}  last_mid={last_mid:.2f}  pnl={pnl:+,.2f}")

print(f"\nTOTAL PnL across all 5 products: {total_pnl:+,.2f}")


GALAXY_SOUNDS_DARK_MATTER: cash=+495,342.00  net_qty=-45  last_mid=10264.50  pnl=+33,439.50
GALAXY_SOUNDS_BLACK_HOLES: cash=+549,641.00  net_qty=-45  last_mid=13457.50  pnl=-55,946.50
GALAXY_SOUNDS_PLANETARY_RINGS: cash=+448,672.00  net_qty=-45  last_mid=9648.50  pnl=+14,489.50
GALAXY_SOUNDS_SOLAR_WINDS: cash=+439,669.00  net_qty=-45  last_mid=10247.50  pnl=-21,468.50
GALAXY_SOUNDS_SOLAR_FLAMES: cash=+493,557.00  net_qty=-45  last_mid=10823.00  pnl=+6,522.00

TOTAL PnL across all 5 products: -22,964.00


In [41]:
# total buy and sell quantities across all 5 products
def classify(df):
    side = pd.Series("mid", index=df.index)
    side[df["price"] >= df["ask_price_1"]] = "buy"
    side[df["price"] <= df["bid_price_1"]] = "sell"
    return df.assign(side=side)

total_buy_qty = 0
total_sell_qty = 0
for p in PRODUCTS:
    df = classify(trades_by_product[p])
    buy_qty = int(df.loc[df["side"] == "buy", "quantity"].sum())
    sell_qty = int(df.loc[df["side"] == "sell", "quantity"].sum())
    total_buy_qty += buy_qty
    total_sell_qty += sell_qty
    print(f"{p}: buys={buy_qty}  sells={sell_qty}")

print(f"\nTotal buy quantity:  {total_buy_qty}")
print(f"Total sell quantity: {total_sell_qty}")


GALAXY_SOUNDS_DARK_MATTER: buys=880  sells=925
GALAXY_SOUNDS_BLACK_HOLES: buys=880  sells=925
GALAXY_SOUNDS_PLANETARY_RINGS: buys=880  sells=925
GALAXY_SOUNDS_SOLAR_WINDS: buys=880  sells=925
GALAXY_SOUNDS_SOLAR_FLAMES: buys=880  sells=925

Total buy quantity:  4400
Total sell quantity: 4625


In [42]:
# spread (ask_1 - bid_1) at the moment each trade occurred, per product
for p in PRODUCTS:
    df = trades_by_product[p].copy()
    df["spread"] = df["ask_price_1"] - df["bid_price_1"]
    print(f"\n{p}:")
    print(df["spread"].value_counts().sort_index().to_string())
    print(f"  mean: {df['spread'].mean():.2f}  median: {df['spread'].median():.2f}  n_trades: {len(df)}")



GALAXY_SOUNDS_DARK_MATTER:
spread
6      10
7      14
8       1
12     47
13    421
14    229
15     11
  mean: 13.06  median: 13.00  n_trades: 733

GALAXY_SOUNDS_BLACK_HOLES:
spread
6       4
7      11
8       7
9       3
12      4
13     75
14    268
15    161
16    140
17     44
18     16
  mean: 14.53  median: 14.00  n_trades: 733

GALAXY_SOUNDS_PLANETARY_RINGS:
spread
6       4
7      16
8       5
12     57
13    183
14    221
15    213
16     34
  mean: 13.74  median: 14.00  n_trades: 733

GALAXY_SOUNDS_SOLAR_WINDS:
spread
5       2
6       7
7      10
8       6
12     54
13    272
14    319
15     63
  mean: 13.32  median: 14.00  n_trades: 733

GALAXY_SOUNDS_SOLAR_FLAMES:
spread
6       3
7      15
8       7
13     63
14    406
15    217
16     22
  mean: 14.04  median: 14.00  n_trades: 733


In [43]:
# spread (ask_1 - bid_1) at every trade timestamp, one column per product
spread_df = pd.DataFrame({
    p: trades_by_product[p].assign(spread=lambda d: d["ask_price_1"] - d["bid_price_1"])
       .set_index(["day", "timestamp"])["spread"]
    for p in PRODUCTS
})
spread_df


GALAXY_SOUNDS_DARK_MATTER  GALAXY_SOUNDS_BLACK_HOLES  \
day timestamp                                                         
2   1700                              12                         13   
    14500                             13                         13   
    15100                             13                         13   
    26500                             13                         12   
    36400                             12                         13   
...                                  ...                        ...   
4   994600                            14                         17   
    995000                            13                         17   
    995100                            13                         17   
    998900                             7                          9   
    999400                            14                         17   

               GALAXY_SOUNDS_PLANETARY_RINGS  GALAXY_SOUNDS_SOLAR_WINDS  \
day timestamp                                                             
2   1700                                  13                         13   
    14500                                 14                         13   
    15100                                 13                         13   
    26500                                 13                         13   
    36400                                 14                         13   
...                                      ...                        ...   
4   994600                                13                         14   
    995000                                13                         14   
    995100                                13                         14   
    998900                                 6                          8   
    999400                                13                         14   

               GALAXY_SOUNDS_SOLAR_FLAMES  
day timestamp                              
2   1700                               13  
    14500                              13  
    15100                              13  
    26500                              13  
    36400                              13  
...                                   ...  
4   994600                             14  
    995000                             14  
    995100                             14  
    998900                              8  
    999400                             14  

[733 rows x 5 columns]

In [46]:
spread_df[spread_df["GALAXY_SOUNDS_DARK_MATTER"] <= 8].head(15)

GALAXY_SOUNDS_DARK_MATTER  GALAXY_SOUNDS_BLACK_HOLES  \
day timestamp                                                         
2   91700                              7                          7   
    109600                             7                          6   
    206200                             7                          7   
    265200                             6                          6   
    804500                             7                          8   
    891600                             6                          7   
3   25600                              7                          7   
    174100                             7                          8   
    230300                             6                          7   
    312000                             6                          6   
    377800                             6                          6   
    380200                             7                          7   
    719500                             6                          7   
    783800                             6                          7   
    813500                             7                          7   

               GALAXY_SOUNDS_PLANETARY_RINGS  GALAXY_SOUNDS_SOLAR_WINDS  \
day timestamp                                                             
2   91700                                  8                          7   
    109600                                 7                          7   
    206200                                 7                          8   
    265200                                 6                          6   
    804500                                 7                          6   
    891600                                 7                          5   
3   25600                                  7                          5   
    174100                                 8                          6   
    230300                                 6                          6   
    312000                                 6                          6   
    377800                                 7                          7   
    380200                                 7                          7   
    719500                                 7                          7   
    783800                                 8                          6   
    813500                                 7                          7   

               GALAXY_SOUNDS_SOLAR_FLAMES  
day timestamp                              
2   91700                               7  
    109600                              7  
    206200                              8  
    265200                              7  
    804500                              7  
    891600                              6  
3   25600                               7  
    174100                              8  
    230300                              7  
    312000                              7  
    377800                              7  
    380200                              8  
    719500                              6  
    783800                              7  
    813500                              7

In [48]:
low_spread_idx = spread_df[spread_df["GALAXY_SOUNDS_DARK_MATTER"] <= 8].head(15).index

rows = []
for (day, ts), _ in [((d, t), None) for d, t in low_spread_idx]:
    for p in PRODUCTS:
        tr = trades_by_product[p]
        tr_row = tr[(tr["day"] == day) & (tr["timestamp"] == ts)]
        if tr_row.empty:
            continue
        tr_row = tr_row.iloc[0]
        bid = tr_row["bid_price_1"]
        ask = tr_row["ask_price_1"]
        price = tr_row["price"]
        side = "buy" if price >= ask else ("sell" if price <= bid else "mid")

        # previous timestamp on same day from the orderbook
        prev = frames[p][(frames[p]["day"] == day) & (frames[p]["timestamp"] < ts)].tail(1)
        prev_bid = prev["bid_price_1"].iloc[0] if not prev.empty else None
        prev_ask = prev["ask_price_1"].iloc[0] if not prev.empty else None

        rows.append({
            "day": day, "timestamp": ts, "product": p,
            "trade_price": price, "qty": tr_row["quantity"],
            "bid": bid, "ask": ask,
            "prev_bid": prev_bid, "prev_ask": prev_ask,
            "side": side,
        })

low_spread_view = pd.DataFrame(rows)
low_spread_view.head(20)


,day,timestamp,product,trade_price,qty,bid,ask,prev_bid,prev_ask,side
0,2,91700,GALAXY_SOUNDS_DARK_MATTER,9870.0,1,9870,9877,9860,9873,sell
1,2,91700,GALAXY_SOUNDS_BLACK_HOLES,9953.0,1,9953,9960,9943,9955,sell
2,2,91700,GALAXY_SOUNDS_PLANETARY_RINGS,10219.0,1,10219,10227,10226,10239,sell
3,2,91700,GALAXY_SOUNDS_SOLAR_WINDS,10201.0,1,10201,10208,10177,10191,sell
4,2,91700,GALAXY_SOUNDS_SOLAR_FLAMES,10419.0,1,10419,10426,10422,10436,sell
5,2,109600,GALAXY_SOUNDS_DARK_MATTER,9874.0,4,9867,9874,9854,9867,buy
6,2,109600,GALAXY_SOUNDS_BLACK_HOLES,9929.0,4,9923,9929,9907,9919,buy
7,2,109600,GALAXY_SOUNDS_PLANETARY_RINGS,10383.0,4,10376,10383,10362,10376,buy
8,2,109600,GALAXY_SOUNDS_SOLAR_WINDS,10097.0,4,10090,10097,10097,10110,buy
9,2,109600,GALAXY_SOUNDS_SOLAR_FLAMES,10583.0,4,10576,10583,10568,10581,buy


In [50]:
import plotly.graph_objects as go
import plotly.colors as pc
from scipy import stats
import numpy as np

colors = pc.qualitative.Plotly

def plot_diff_distribution(col, title):
    fig = go.Figure()
    for i, p in enumerate(PRODUCTS):
        diffs = frames[p].groupby("day")[col].diff().dropna()
        if diffs.empty:
            continue
        color = colors[i % len(colors)]
        mu, sigma = diffs.mean(), diffs.std()

        # histogram (normalised to density so the normal PDF overlays correctly)
        fig.add_trace(go.Histogram(
            x=diffs, name=f"{p} hist",
            legendgroup=p, marker_color=color,
            opacity=0.5, nbinsx=80, histnorm="probability density",
        ))

        # normal pdf curve fitted to the same data
        xs = np.linspace(diffs.min(), diffs.max(), 400)
        pdf = stats.norm.pdf(xs, loc=mu, scale=sigma)
        fig.add_trace(go.Scatter(
            x=xs, y=pdf, mode="lines",
            name=f"{p} N({mu:+.3f}, {sigma:.3f})",
            legendgroup=p, line=dict(color=color, width=2),
        ))

        print(f"{p}: mean={mu:+.4f}  std={sigma:.4f}  n={len(diffs)}")

    fig.update_layout(
        barmode="overlay",
        title=title,
        xaxis_title=f"{col}_t - {col}_{{t-1}}",
        yaxis_title="density",
        legend=dict(groupclick="togglegroup"),
    )
    fig.show(renderer="browser")

plot_diff_distribution("mid_price", "Mid-price first-difference distribution + normal fit")


GALAXY_SOUNDS_DARK_MATTER: mean=+0.0089  std=10.2457  n=29997
GALAXY_SOUNDS_BLACK_HOLES: mean=+0.1152  std=11.4797  n=29997
GALAXY_SOUNDS_PLANETARY_RINGS: mean=-0.0120  std=10.8791  n=29997
GALAXY_SOUNDS_SOLAR_WINDS: mean=+0.0084  std=10.5392  n=29997
GALAXY_SOUNDS_SOLAR_FLAMES: mean=+0.0277  std=11.0946  n=29997


In [51]:
plot_diff_distribution("bid_price_1", "Best-bid first-difference distribution + normal fit")


GALAXY_SOUNDS_DARK_MATTER: mean=+0.0089  std=10.2693  n=29997
GALAXY_SOUNDS_BLACK_HOLES: mean=+0.1151  std=11.5069  n=29997
GALAXY_SOUNDS_PLANETARY_RINGS: mean=-0.0120  std=10.9041  n=29997
GALAXY_SOUNDS_SOLAR_WINDS: mean=+0.0083  std=10.5693  n=29997
GALAXY_SOUNDS_SOLAR_FLAMES: mean=+0.0277  std=11.1148  n=29997


In [52]:
plot_diff_distribution("ask_price_1", "Best-ask first-difference distribution + normal fit")


GALAXY_SOUNDS_DARK_MATTER: mean=+0.0090  std=10.2998  n=29997
GALAXY_SOUNDS_BLACK_HOLES: mean=+0.1153  std=11.5376  n=29997
GALAXY_SOUNDS_PLANETARY_RINGS: mean=-0.0120  std=10.9349  n=29997
GALAXY_SOUNDS_SOLAR_WINDS: mean=+0.0084  std=10.5866  n=29997
GALAXY_SOUNDS_SOLAR_FLAMES: mean=+0.0278  std=11.1566  n=29997


In [56]:
# normal mid (level 1) vs wall mid (level 2), and spread distributions for both
for p in PRODUCTS:
    df = frames[p].copy()
    df["mid_1"] = (df["bid_price_1"] + df["ask_price_1"]) / 2
    df["mid_2"] = (df["bid_price_2"] + df["ask_price_2"]) / 2
    df["spread_1"] = df["ask_price_1"] - df["bid_price_1"]
    df["spread_2"] = df["ask_price_2"] - df["bid_price_2"]

    print(f"\n=== {p} ===")
    print(f"normal mid (lvl 1): mean={df['mid_1'].mean():.2f}  std={df['mid_1'].std():.2f}  n={df['mid_1'].notna().sum()}")
    print(f"wall mid   (lvl 2): mean={df['mid_2'].mean():.2f}  std={df['mid_2'].std():.2f}  n={df['mid_2'].notna().sum()}")

    print("\nspread_1 (ask_1 - bid_1) counts:")
    print(df["spread_1"].value_counts().sort_index().to_string())
    print("\nspread_2 (ask_2 - bid_2) counts:")
    print(df["spread_2"].value_counts().sort_index().to_string())



=== GALAXY_SOUNDS_DARK_MATTER ===
normal mid (lvl 1): mean=10226.66  std=330.70  n=30000
wall mid   (lvl 2): mean=10226.67  std=330.70  n=30000

spread_1 (ask_1 - bid_1) counts:
spread_1
5        10
6       419
7       498
8        70
12     1471
13    18534
14     8649
15      349

spread_2 (ask_2 - bid_2) counts:
spread_2
14.0      294
15.0     2398
16.0    16791
17.0     9467
18.0     1050

=== GALAXY_SOUNDS_BLACK_HOLES ===
normal mid (lvl 1): mean=11466.87  std=958.44  n=30000
wall mid   (lvl 2): mean=11466.88  std=958.45  n=30000

spread_1 (ask_1 - bid_1) counts:
spread_1
5         2
6       147
7       399
8       356
9        85
10        8
12      281
13     3428
14    10436
15     6663
16     5921
17     1648
18      626

spread_2 (ask_2 - bid_2) counts:
spread_2
14.0      33
15.0     527
16.0    2837
17.0    8304
18.0    7356
19.0    4366
20.0    4875
21.0     933
22.0     766
23.0       3

=== GALAXY_SOUNDS_PLANETARY_RINGS ===
normal mid (lvl 1): mean=10766.67  std=765.84  

In [58]:
# for each product, when spread_1 is low (<= some threshold), what does spread_2 look like?
LOW_SPREAD_THRESH = 7

for p in PRODUCTS:
    df = frames[p].copy()
    df["spread_1"] = df["ask_price_1"] - df["bid_price_1"]
    df["spread_2"] = df["ask_price_2"] - df["bid_price_2"]
    low = df[df["spread_1"] <= LOW_SPREAD_THRESH]

    print(f"\n=== {p} ===  (rows with spread_1 <= {LOW_SPREAD_THRESH}: {len(low)} / {len(df)})")
    if low.empty:
        continue

    # crosstab: spread_1 along rows, spread_2 along columns
    ct = pd.crosstab(low["spread_1"], low["spread_2"], dropna=False)
    print(ct.to_string())

    # quick summary of spread_2 in this regime
    s2 = low["spread_2"]
    print(f"\nspread_2 when spread_1 <= {LOW_SPREAD_THRESH}: "
          f"mean={s2.mean():.2f}  median={s2.median():.2f}  "
          f"missing={s2.isna().sum()}  n={s2.notna().sum()}")



=== GALAXY_SOUNDS_DARK_MATTER ===  (rows with spread_1 <= 7: 927 / 30000)
spread_2  14.0  15.0  16.0
spread_1                  
5           10     0     0
6          146   237    36
7          138   307    53

spread_2 when spread_1 <= 7: mean=14.78  median=15.00  missing=0  n=927

=== GALAXY_SOUNDS_BLACK_HOLES ===  (rows with spread_1 <= 7: 548 / 30000)
spread_2  14.0  15.0  16.0  17.0  18.0
spread_1                              
5            2     0     0     0     0
6           12    47    88     0     0
7           19    89   157    84    50

spread_2 when spread_1 <= 7: mean=15.97  median=16.00  missing=0  n=548

=== GALAXY_SOUNDS_PLANETARY_RINGS ===  (rows with spread_1 <= 7: 752 / 30000)
spread_2  13.0  14.0  15.0  16.0  17.0  18.0
spread_1                                    
5            0    17     0     0     0     0
6            5    89    83    77     0     0
7            9    80   138   146   102     6

spread_2 when spread_1 <= 7: mean=15.31  median=15.00  missing=0  n=7